# Chapter 3 publication and verification

Run after the completed numerical notebook. This notebook creates release evidence and Earth Engine assets; browser candidate testing and live publication remain separate required steps.

In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive', timeout_ms=240000)
auth.authenticate_user()
from pathlib import Path
import sys, json, hashlib, zipfile, importlib, importlib.metadata as metadata, subprocess
ROOT=Path('/content/drive/MyDrive/PHD/CSP3_GPP_outputs')
OUTPUT=ROOT/'CH3_profitability/20261007_profitability_v1'
def installed_version(name):
    try: return metadata.version(name)
    except metadata.PackageNotFoundError: return None
before={line.split('==')[0]:installed_version(line.split('==')[0]) for line in (OUTPUT/'code/requirements.txt').read_text().splitlines() if '==' in line}
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(OUTPUT/'code/requirements.txt')])
for line in (OUTPUT/'code/requirements.txt').read_text().splitlines():
    if '==' in line:
        name,version=line.split('=='); assert metadata.version(name)==version
print('Drive and pinned dependencies ready')
if any(metadata.version(name)!=version for name,version in before.items()):
    raise RuntimeError('Pinned dependencies were installed. Use Runtime > Restart session and run all once before analysis.')


In [ ]:
archive=OUTPUT/'publication_package_01cb740f4a88c5c9.zip'
package_sha='01cb740f4a88c5c9f9945a2386bc022f22c338a450c1a3bfefbe5bbd84796da6'
assert hashlib.sha256(archive.read_bytes()).hexdigest()==package_sha
CODE=Path('/content/ch3_publication_'+package_sha[:16])
with zipfile.ZipFile(archive) as z: z.extractall(CODE)
manifest=json.loads((CODE/'publication_manifest.json').read_text())
for row in manifest:
    assert hashlib.sha256((CODE/row['file']).read_bytes()).hexdigest()==row['sha256'],row['file']
for name in list(sys.modules):
    if name.startswith(('ch3_profitability','ch4_marginality')): del sys.modules[name]
sys.path.insert(0,str(CODE)); importlib.invalidate_caches()
from ch4_marginality.profit_workspace import lock_inputs
from ch3_profitability.pipeline import SOURCE_RELEASE
prior=ROOT/'CH4_marginality'/SOURCE_RELEASE
lock_inputs(ROOT,prior/'analysis',OUTPUT,json.loads((prior/'input_manifest.json').read_text()))
(OUTPUT/'publication_package_record.json').write_text(json.dumps({'archive':archive.name,'sha256':package_sha,'files':len(manifest)},indent=2))
print('Verified immutable helper package and current scientific inputs')

In [ ]:
from ch3_profitability.recover_release import run as recover_release
record=OUTPUT/'verification/cloud_write_recovery.json'
if not record.exists() or not json.loads(record.read_text()).get('closed_sqlite_connection'):
    print(recover_release(ROOT,OUTPUT),flush=True)
from ch3_profitability.audit_release import run as audit
print(audit(OUTPUT),flush=True)

In [ ]:
import ee
from google.cloud import storage
versions={name:metadata.version(name) for name in ['earthengine-api','google-cloud-storage','numpy','pandas','rasterio','geopandas']}
(OUTPUT/'publication_environment.json').write_text(json.dumps(versions,indent=2))
try:
    ee.Initialize(project='ee-njberkowitz95')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='ee-njberkowitz95')
print('Earth Engine connected; dependency versions recorded')

In [ ]:
from ch3_profitability.assets import prepare
prepare(OUTPUT)
print('Earth Engine tables and temporal image prepared')

In [ ]:
from ch3_profitability.assets import submit
tasks=submit(OUTPUT)
print(json.dumps(tasks,indent=2))

In [ ]:
from ch3_profitability.supplement import run as figures
from ch3_profitability.report import run as report
from ch3_profitability.maps import run as maps
from ch3_profitability.identity import run as identities
figures(OUTPUT)
maps(OUTPUT)
report(OUTPUT)
identities(ROOT,OUTPUT)
print('Publication figures, report and preserved identities complete')

In [ ]:
import time
for attempt in range(60):
    ids=[x['response']['id'] for x in tasks if 'response' in x]
    states=ee.data.getTaskStatus(ids) if ids else []
    print([(x['state'],x.get('error_message')) for x in states],flush=True)
    if any(x['state'] in ['FAILED','CANCELLED'] for x in states): raise RuntimeError('An Earth Engine import failed')
    if all(x['state']=='COMPLETED' for x in states): break
    time.sleep(10)
else: raise RuntimeError('Imports still running; repeat this status cell later')

In [ ]:
from ch3_profitability.assets import verify
from ch3_profitability.verify_native import run as verify_native
verify(OUTPUT)
verify_native(ROOT)
print('New assets and reused native input assets verified')

In [ ]:
from ch4_marginality.profit_workspace import lock_inputs
from ch3_profitability.pipeline import SOURCE_RELEASE
from ch3_profitability.build_app import build
prior=ROOT/'CH4_marginality'/SOURCE_RELEASE
lock_inputs(ROOT,prior/'analysis',OUTPUT,json.loads((prior/'input_manifest.json').read_text()))
candidate=build(OUTPUT)
print('Verified app candidate:',candidate)
print('Browser testing and publication are still required; this notebook does not declare the live app complete.')